<a href="https://www.kaggle.com/code/fernandosilvestresci/notebookd89c50c368?scriptVersionId=306782193" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import kagglehub
import torch
from torchvision import transforms
from  torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np

In [2]:
# Download latest version
path = kagglehub.dataset_download("adilshamim8/people-detection")

print("Path to dataset files:", path)

Path to dataset files: /kaggle/input/datasets/adilshamim8/people-detection


In [3]:
annotation = [item  for item in os.listdir(path + '/train/train') if item.endswith('.csv')]
print(annotation)


['_annotations.csv']


In [4]:
df = pd.read_csv(path + '/train/train/_annotations.csv')
df.head()

,filename,width,height,class,xmin,ymin,xmax,ymax
0,2008_003132_jpg.rf.92f6223defec4f57f2d7b9cfa28...,500,375,person,219,98,269,283
1,2008_003132_jpg.rf.92f6223defec4f57f2d7b9cfa28...,500,375,person,114,124,155,263
2,2008_003132_jpg.rf.92f6223defec4f57f2d7b9cfa28...,500,375,person,43,139,98,340
3,004574_jpg.rf.7c8cea69d7be45f58febcede26ef0c6e...,500,333,person,145,118,229,333
4,004574_jpg.rf.7c8cea69d7be45f58febcede26ef0c6e...,500,333,person,285,105,349,329


In [5]:
class CustomDataset(Dataset):    
    def __init__(self, root_dir, transform=None):
        self._root_dir = root_dir
        self._transform = transform
        images = [item for item in os.listdir(root_dir) if not item.endswith('.csv')]
        self._images = images
        annotations = [item for item in os.listdir(root_dir) if item.endswith('.csv')]
        self._annotations = pd.read_csv(os.path.join(root_dir, annotations[0]))
        self._grouped = self._annotations.groupby('filename')
        

    def __len__(self):
        return len(self._images)

    def __getitem__(self, idx):
        
        filename = self._images[idx]
        
        image_path = os.path.join(self._root_dir, filename)
        image = Image.open(image_path).convert('RGB')

        boxes = []
        labels = []

        if filename in self._grouped.groups:
            dt = self._grouped.get_group(filename)

            for _, row in dt.iterrows():
                if row['class'] != 'person':
                    continue

                boxes.append([row['xmin'], row['ymin'], row['xmax'], row['ymax']])
                labels.append(0)

        boxes = torch.tensor(boxes, dtype=torch.float32)
        labels = torch.tensor(labels, dtype=torch.int64)

        if len(boxes) == 0:
            boxes = torch.zeros((0, 4), dtype=torch.float32)
            labels = torch.zeros((0,), dtype=torch.int64)

        target = {
            "boxes": boxes,
            "labels": labels
        }

        if self._transform:
            image = self._transform(image)

        return image, target
        
        

In [6]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

valid_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],std=[0.229, 0.224, 0.225])
])

In [7]:
def collate_fn(batch):
    return tuple(zip(*batch))

BATCH_SIZE=32


train_dataset = CustomDataset(path + '/train/train', train_transform)
valid_dataset = CustomDataset(path + '/valid/valid', valid_transform)

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn
)

valid_loader = DataLoader(
    valid_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn
)

In [8]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv = nn.Sequential(
            nn.Conv2d(3, 16, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(16, 32, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            nn.Conv2d(32, 64, 3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.fc = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 28 * 28, 128),
            nn.ReLU(),
            nn.Linear(128, 2)
        )

    def forward(self, x):
        x = self.conv(x)
        x = x.view(x.size(0), -1)
        x = self.fc(x)
        return x
        

In [9]:
model = CNN()
model = model.to('cpu')

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [10]:
def extract_labels(targets):
    """Pega o primeiro label de cada target, ou 0 se não houver pessoas."""
    result = []
    for t in targets:
        if len(t['labels']) > 0:
            result.append(t['labels'][0])
        else:
            result.append(torch.tensor(0, dtype=torch.int64))
    return torch.stack(result)

In [11]:
def evaluate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    running_correct = 0
    total = 0

    with torch.no_grad():
        for images, targets in loader:
            images = torch.stack(images).to(device)
            labels = extract_labels(targets).to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * labels.size(0)
            preds = outputs.argmax(1)
            running_correct += (preds == labels).sum().item()
            total += labels.size(0)

    return running_loss / total, running_correct / total

In [12]:
device = 'cpu'
EPOCHES = 20

scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, 
    mode='max',
    factor=0.5,
    patience=3
)

history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
    'lr': []
}

best_val_acc = 0.0
epochs_no_improve = 0
patiences = 5

print("Starting training...")


for epoch in range(EPOCHES):
    print(f"Epoch {epoch+1}/{EPOCHES}")
    model.train()
    running_loss = 0.0
    running_correct = 0
    total = 0


    for images, targets in train_loader:
        images = torch.stack(images).to(device)
        labels = extract_labels(targets).to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * labels.size(0)
        preds = outputs.argmax(1)
        running_correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_loss = running_loss / total
    train_acc = running_correct / total

    val_loss, val_acc = evaluate(model, valid_loader, criterion, device)

    scheduler.step(val_acc)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    history['lr'].append(optimizer.param_groups[0]['lr'])
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.4f} - "
          f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.4f} - LR: {optimizer.param_groups[0]['lr']:.6f}")

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print("Model saved!")
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if epochs_no_improve >= patiences:
        print("Early stopping triggered!")
        break



Starting training...
Epoch 1/20
Train Loss: 0.0022, Train Acc: 1.0000 - Val Loss: 0.0000, Val Acc: 1.0000 - LR: 0.000100
Model saved!
Epoch 2/20
Train Loss: 0.0000, Train Acc: 1.0000 - Val Loss: 0.0000, Val Acc: 1.0000 - LR: 0.000100
Epoch 3/20
Train Loss: 0.0000, Train Acc: 1.0000 - Val Loss: 0.0000, Val Acc: 1.0000 - LR: 0.000100
Epoch 4/20
Train Loss: 0.0000, Train Acc: 1.0000 - Val Loss: 0.0000, Val Acc: 1.0000 - LR: 0.000100
Epoch 5/20
Train Loss: 0.0000, Train Acc: 1.0000 - Val Loss: 0.0000, Val Acc: 1.0000 - LR: 0.000050
Epoch 6/20
Train Loss: 0.0000, Train Acc: 1.0000 - Val Loss: 0.0000, Val Acc: 1.0000 - LR: 0.000050
Early stopping triggered!


In [13]:
model.eval()

CNN(
  (conv): Sequential(
    (0): Conv2d(3, 16, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(16, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fc): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=50176, out_features=128, bias=True)
    (2): ReLU()
    (3): Linear(in_features=128, out_features=2, bias=True)
  )
)

In [14]:
torch.save(model.state_dict(), "/kaggle/working/model.pth")